In [ ]:
import serial
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
class ImagePacketDecoder:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.frame_size = width * height * 2   # 每像素 2 bytes (MSB first)

        self.state = "WAIT_HEADER1"
        self.addr = 0
        self.buffer = np.zeros(self.frame_size, dtype=np.uint8)

    def reset_frame(self):
        self.state = "WAIT_HEADER1"
        self.addr = 0

    def feed_byte(self, byte):
        if self.state == "WAIT_HEADER1":
            if byte == 0xFF:
                self.state = "WAIT_HEADER2"
            return None

        elif self.state == "WAIT_HEADER2":
            if byte == 0xFF:
                self.state = "RECEIVE_DATA"
            else:
                self.state = "WAIT_HEADER1"
            return None

        elif self.state == "RECEIVE_DATA":
            if self.addr < self.frame_size:
                self.buffer[self.addr] = byte
                self.addr += 1

            if self.addr >= self.frame_size:
                # 解析為 16-bit MSB-first
                raw16 = (self.buffer[0::2].astype(np.uint16) << 8) | self.buffer[1::2]
                img16 = raw16.reshape((self.height, self.width))

                # 轉成 8-bit 顯示
                img8 = (img16 >> 8).astype(np.uint8)

                self.reset_frame()
                return img8, img16, raw16


def uart_reader_once(ser, decoder):
    """读取一次 UART，如果收到一帧就返回图像"""
    raw = ser.read(1)
    if not raw:
        return None

    byte = raw[0]
    return decoder.feed_byte(byte)

def load_calibration_frame(path, width, height):
    with open(path, "r") as f:
        hex_str = f.read().strip()

    hex_values = hex_str.split()
    raw16 = np.array([int(h, 16) for h in hex_values], dtype=np.uint16)
    calib_img16 = raw16.reshape((height, width))
    return calib_img16

def run_uart_in_notebook(port="COM8", baud=115200, width=160, height=120, enable_calibration=True):
    #ser = serial.Serial(port, baud, timeout=0.01)
    with serial.Serial(port, baud, timeout=0.01) as ser:
        decoder = ImagePacketDecoder(width, height)

        if enable_calibration:
            calib_img16 = load_calibration_frame("1-frame.txt", width, height)
    
        print("等待圖像幀...")
    
        byte_count = 0
    
        while True:
            raw = ser.read(1)
            if raw:
                byte_count += 1
                result = decoder.feed_byte(raw[0])
            else:
                continue
            
            if result is not None:
                img8, img16, raw16 = result
    
                if enable_calibration:
                    corrected16 = img16.astype(np.int32) - calib_img16.astype(np.int32)
                    corrected16 = np.clip(corrected16, 0, 65535).astype(np.uint16)
                    corrected8 = (corrected16 >> 8).astype(np.uint8)
                    display_img = corrected16
                else:
                    display_img = img16
    
                clear_output(wait=True)
                print("display_img shape =", display_img.shape, "dtype =", display_img.dtype)
                plt.imshow(display_img, cmap="gray")
                plt.title("Received Image")
                plt.show()
    
                print("收到一幀圖像")
                print(f"本幀接收到的字節數：{byte_count}")

                # 顯示 RAW 全部資料（16-bit HEX）
                hex_full = " ".join(f"{b:04X}" for b in raw16)
                print("RAW 全部資料（16-bit HEX）：")
                print(hex_full)

                byte_count = 0


run_uart_in_notebook("COM31", 5000000, 160, 120, enable_calibration=True)

SerialException: could not open port 'COM31': PermissionError(13, '拒绝访问。', None, 5)